In [7]:
import sys 
sys.path.append("../")
import pandas as pd 
import numpy as np 
import datetime as dt
from technical.indicators import BollingerBands


In [4]:
df_raw = pd.read_pickle("../data/EUR_USD_H1.pkl")

In [6]:
df_an = df_raw.copy()
df_an.reset_index(drop=True, inplace=True)

In [23]:
def calculate_bollinger_bands(data, window, std):
    rolling_mean = data["close"].rolling(window=window).mean()
    rolling_std = data["close"].rolling(window=window).std()
    data["upperband"] = rolling_mean + std * rolling_std
    data["lowerband"] = rolling_mean - std * rolling_std
    return data
window = 20
std = 2.0

In [24]:
#df_an = BollingerBands(df_an, window, std)
data = calculate_bollinger_bands(df_an, window, std)


KeyError: 'close'

In [19]:
df_an["Signal"] = None
df_an["Position"] = None
data = df_an

In [17]:
df_an

,time,volume,mid_o,mid_h,mid_l,mid_c,bid_o,bid_h,bid_l,bid_c,ask_o,ask_h,ask_l,ask_c,BB_MA,BB_UP,BB_LW,Signal,Position
0,2015-06-01 00:00:00+00:00,1195,1.09573,1.09628,1.09346,1.09424,1.09565,1.09620,1.09337,1.09415,1.09581,1.09636,1.09355,1.09433,NaN,NaN,NaN,None,None
1,2015-06-01 01:00:00+00:00,922,1.09424,1.09473,1.09305,1.09420,1.09416,1.09465,1.09297,1.09412,1.09433,1.09481,1.09313,1.09427,NaN,NaN,NaN,None,None
2,2015-06-01 02:00:00+00:00,656,1.09418,1.09540,1.09404,1.09509,1.09409,1.09531,1.09395,1.09501,1.09426,1.09549,1.09412,1.09517,NaN,NaN,NaN,None,None
3,2015-06-01 03:00:00+00:00,443,1.09506,1.09556,1.09481,1.09502,1.09498,1.09549,1.09473,1.09495,1.09513,1.09564,1.09489,1.09509,NaN,NaN,NaN,None,None
4,2015-06-01 04:00:00+00:00,813,1.09498,1.09652,1.09491,1.09598,1.09491,1.09644,1.09484,1.09590,1.09505,1.09659,1.09498,1.09605,NaN,NaN,NaN,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59616,2024-12-30 19:00:00+00:00,3925,1.04045,1.04055,1.03962,1.03966,1.04038,1.04048,1.03954,1.03958,1.04052,1.04062,1.03969,1.03973,1.041789,1.045564,1.038014,None,None
59617,2024-12-30 20:00:00+00:00,3987,1.03966,1.04052,1.03952,1.03988,1.03959,1.04045,1.03944,1.03980,1.03973,1.04060,1.03960,1.03995,1.041647,1.045473,1.037821,None,None
59618,2024-12-30 21:00:00+00:00,1875,1.03988,1.04070,1.03984,1.04068,1.03980,1.04062,1.03976,1.04060,1.03996,1.04079,1.03993,1.04076,1.041532,1.045362,1.037701,None,None
59619,2024-12-30 22:00:00+00:00,1018,1.04032,1.04056,1.04014,1.04052,1.03982,1.04042,1.03970,1.04038,1.04082,1.04082,1.04039,1.04066,1.041423,1.045251,1.037595,None,None


In [33]:
stop_loss_pct = 0.05
transaction_cost_pct = 0.005
capital = 1000


In [26]:
for i in range(window, len(data)):
    # check for buy signal
    if data["ask_c"][i] > data["BB_UP"][i-1] and data["ask_c"][i -1] <= data["BB_UP"][i-1]:
        data["Signal"][i] = "Buy"
        data["Position"][i] = 1
        # Calulate stop level
        stop_loss_level = data["ask_c"][i] * (1 - stop_loss_pct)
    # check for short signal
    elif data["bid_c"][i] < data["BB_LW"][i-1] and data["bid_c"][i-1] >= data["BB_LW"][i-1]:
        data["Signal"][i] = "Sell"
        data["Position"][i] = -1
        # Calulate stop loss level 
        stop_loss_level = data["bid_c"][i] * (1 + stop_loss_pct)
    else:
        # check for stop loss 
        if data["Position"][i-1] == 1 and data["bid_c"][i] < stop_loss_level:
            data["Signal"][i] = "Stop Loss"
            data["Position"] = 0
        elif data["Position"][i -1] == -1 and data["bid_c"][i] > stop_loss_level:
            data["Signal"][i] = "Stop Loss"
            data["Position"][i]= 0 
        else:
            data["Position"][i] = data["Position"][i-1]

    #Apply trasnsaction costs 
    if data["Position"][i] != data["Position"][i-1]:
        transaction_cost = abs(data["Position"][i] - data["Position"][i-1]) * transaction_cost_pct * data["bid_c"][i]
        capital -= transaction_cost

    portfolio_value = capital + (data["Position"][i]*data["ask_c"][i])

    # update stop loss level for open positions 
    if data["Position"][i] == 1:
        stop_loss_level = max(stop_loss_level, data["ask_c"][i] * (1-stop_loss_pct))
    elif data["Position"][i] == -1:
        stop_loss_level = min(stop_loss_level, data["ask_c"][i] * (1 + stop_loss_pct))

C:\Users\otavi\AppData\Local\Temp\ipykernel_19956\1307253455.py:23: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  data["Position"][i] = data["Position"][i-1]
C:\Users\otavi\AppData\Local\Temp\ipykernel_19956\1307253455.py:23: SettingWithCopy

TypeError: unsupported operand type(s) for *: 'NoneType' and 'float'

In [34]:
# --- before the loop ---
data["Signal"] = None
data["Position"] = 0.0   # initialize explicitly so i-1 lookups are never NaN
stop_loss_level = None

for i in range(window, len(data)):
    # check for buy signal
    if data["ask_c"][i] > data["BB_UP"][i-1] and data["ask_c"][i-1] <= data["BB_UP"][i-1]:
        data.loc[i, "Signal"] = "Buy"
        data.loc[i, "Position"] = 1
        stop_loss_level = data["ask_c"][i] * (1 - stop_loss_pct)

    # check for short signal
    elif data["bid_c"][i] < data["BB_LW"][i-1] and data["bid_c"][i-1] >= data["BB_LW"][i-1]:
        data.loc[i, "Signal"] = "Sell"
        data.loc[i, "Position"] = -1
        stop_loss_level = data["bid_c"][i] * (1 + stop_loss_pct)

    else:
        # check for stop loss
        if data["Position"][i-1] == 1 and stop_loss_level is not None and data["bid_c"][i] < stop_loss_level:
            data.loc[i, "Signal"] = "Stop Loss"
            data.loc[i, "Position"] = 0
        elif data["Position"][i-1] == -1 and stop_loss_level is not None and data["bid_c"][i] > stop_loss_level:
            data.loc[i, "Signal"] = "Stop Loss"
            data.loc[i, "Position"] = 0
        else:
            data.loc[i, "Position"] = data["Position"][i-1]

    # Apply transaction costs
    if data["Position"][i] != data["Position"][i-1]:
        transaction_cost = abs(data["Position"][i] - data["Position"][i-1]) * transaction_cost_pct * data["bid_c"][i]
        capital -= transaction_cost

    portfolio_value = capital + (data["Position"][i] * data["ask_c"][i])

    # update stop loss level for open positions
    if data["Position"][i] == 1:
        stop_loss_level = max(stop_loss_level, data["ask_c"][i] * (1 - stop_loss_pct))
    elif data["Position"][i] == -1:
        stop_loss_level = min(stop_loss_level, data["ask_c"][i] * (1 + stop_loss_pct))

In [35]:
portfolio_value

np.float64(978.8656074500016)